# 标志物优化

如果你有实验数据（比如药物处理后的基因表达 + 药效数据），可以用这个功能优化你的初始基因表达特征：

In [ ]:
import drugreflector as dr
import pandas as pd

In [ ]:
这里考虑到实际方法的使用 我觉得Starting signature是使用V-score Computation计算得到的基因层面的差异表达分数向量（以基因为索引，值为差异分数）
最经典的就是计算"cell_type:control->treatment"的V-score

In [ ]:
# Starting signature (pandas Series with gene names as index)
starting_signature = pd.Series([1.2, -0.8, 0.5, ...], 
                              index=['GENE1', 'GENE2', 'GENE3', ...])

In [ ]:
# Initialize signature refinement (available at top level)
refiner = dr.SignatureRefinement(starting_signature)

## 加载实验数据将单细胞或批量 RNA-seq 数据按 化合物 + 样本 聚合。  
如果指定了 signature_id_obs_cols=['treatment_type', 'timepoint']，则会为每种组合（如 "IFN-gamma + 24h"）单独建模。

In [ ]:
# Load experimental data (AnnData with compound treatments)
# adata should have:
# - Gene expression data in .X or layers
# - Compound IDs in .obs (e.g., 'compound_id' column)
# - Sample IDs in .obs (e.g., 'sample_id' column) 
refiner.load_counts_data(
    adata, 
    compound_id_obs_col='compound_id',
    sample_id_obs_cols=['sample_id'],
    layer='raw_counts'  # or None to use .X
)

## 关联每个化合物的表型值

In [ ]:
# Load phenotypic readouts
readouts = pd.Series([0.8, -1.2, 0.3, ...], 
                    index=['compound_A', 'compound_B', 'compound_C', ...])
refiner.load_phenotypic_readouts(readouts)

## 核心算法：对每个基因，计算它在所有化合物上的表达值 与 表型读数之间的相关性（Pearson 或 Spearman）。  
得到一个新的签名：learned_signature[gene] = corr(gene_expression_across_compounds, phenotype)  
这个签名完全由你的数据驱动，反映哪些基因的表达最能预测表型。  

In [ ]:
# Compute learned signatures using correlation analysis
refiner.compute_learned_signatures(corr_method='pearson')

## 将 初始特征 和 学习到的特征 进行加权融合：

In [ ]:
# Generate refined signatures (interpolation between starting and learned)
refiner.compute_refined_signatures(
    learning_rate=0.5,      # 0.5 = equal weight to starting and learned
    scale_learned_sig=True  # Scale learned signature to match starting signature std
)

## 输出结果

In [ ]:
# Access results
refined_signatures = refiner.refined_signatures  # AnnData object
learned_signatures = refiner.learned_signatures   # AnnData object

本质：一种 数据驱动的签名校准（calibration）方法，平衡先验知识与实验证据。

# Signature Refinement with Multiple Conditions

In [ ]:
# For multiple experimental conditions, specify signature_id_obs_cols
refiner.load_counts_data(
    adata,
    compound_id_obs_col='compound_id',
    sample_id_obs_cols=['sample_id'],
    signature_id_obs_cols=['treatment_type', 'timepoint'],  # Creates separate signatures
    layer='raw_counts'
)

# This will create one learned/refined signature for each unique combination
# of values in signature_id_obs_cols